In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
from IPython.display import Markdown, display

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging, short_id
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from foundry.repository import TrialBalanceRepository
from foundry.config.settings import POSTGRES_TABLE_LOCATIONS

from atlas import AtlasClient

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools, AtlasTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord, BreakAnalysisResult
from break_analysis.builder import BreakCaseBuilder
from break_analysis.services import AtlasEvidenceService, ReconBreakRecordResolver


configure_logging()

def display_df(df):
    display(df.toPandas())

_STATUS_EMOJI = {
    'EXPLAINED': '✅',
    'UNEXPLAINED': '❌',
}

def display_break_result(result: BreakAnalysisResult) -> None:
    emoji = _STATUS_EMOJI.get(str(result.status), '')
    record_ids = ', '.join(short_id(rid) for rid in result.recon_result_ids)

    display(Markdown(
        f"## Break Case `{short_id(result.case_id)}` — {emoji} **{result.status}**\n\n"
        f"**Records investigated ({len(result.recon_result_ids)}):** {record_ids}\n\n"
        f"**Explanation:** {result.explanation}"
    ))

    if not result.findings:
        display(Markdown('_No findings._'))
        return

    display(Markdown(f'**Findings ({len(result.findings)}):**'))
    display_df_findings = pd.DataFrame([
        {
            'recon_result_id': short_id(finding.recon_result_id),
            'segment_type': str(finding.segment_type),
            'segment_value': finding.segment_value or '(blank)',
            'root_cause': str(finding.root_cause),
            'explanation': finding.explanation,
        }
        for finding in result.findings
    ])
    display(display_df_findings)

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/19 21:54:40 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/19 21:54:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-94bc9ae2-bc78-44c6-bff7-ba13ebec1461;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 66ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

atlas_client = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

atlas_evidence_service = AtlasEvidenceService(
    atlas_client=atlas_client, 
    foundry_repository=repository
)

recon_record_resolver = ReconBreakRecordResolver(recon_client=recon)

atlas_tools = AtlasTools(
    recon_resolver=recon_record_resolver,
    atlas_evidence_service=atlas_evidence_service
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    atlas_tools=atlas_tools,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = 'd3282a32-5edc-4d85-b48c-e0ea38d9a4de'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

2026-09-19 21:54:44,961 | INFO | break_analysis.builder | Building break cases | records=7
2026-09-19 21:54:44,962 | INFO | break_analysis.builder | Break cases built | total=3 | one_to_one=2 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1


In [7]:
break_case = break_cases[0]

display_break_result(agent.analyze(break_case))

2026-09-19 21:54:44,965 | INFO | break_analysis.agent | Analyzing break case | case_id=50919c5f | workflow_run_id=d3282a32 | topology=MANY_TO_ONE | records=3
2026-09-19 21:54:44,965 | INFO | break_analysis.agent | Invoking LLM | case_id=50919c5f | round=1
2026-09-19 21:54:53,037 | INFO | break_analysis.agent | LLM invoked | case_id=50919c5f | round=1 | tool_calls=4 | input_tokens=1269 | output_tokens=203 | total_tokens=1472 | prompt_eval_count=1269 | eval_count=203 | load_ms=119 | prompt_eval_ms=895 | eval_ms=6765 | total_ms=8066 | duration_ms=8072
2026-09-19 21:54:53,037 | INFO | break_analysis.agent | Tool round | case_id=50919c5f | round=1 | tool_calls=4
2026-09-19 21:54:53,239 | INFO | break_analysis.agent | Tool invoked | case_id=50919c5f | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "4100000", "business_dt": "2026-03-31"} | duration_ms=201
2026-09-19 21:54:53,348 | INFO | break_analysis.agent | Tool invoked | case_id=50919c5f | tool=validate_segme

## Break Case `50919c5f` — ✅ **EXPLAINED**

**Records investigated (3):** bf31b11b, 73c87bec, 828d5971

**Explanation:** Reconciliation discrepancies caused by missing or inactive GL account and sub-account segments in the registry.

**Findings (4):**

,recon_result_id,segment_type,segment_value,root_cause,explanation
0,73c87bec,GLSegmentType.ACCOUNT,4100000,REGISTRY_INVALID_SEGMENT,Registry record does not exist
1,73c87bec,GLSegmentType.SUB_ACCOUNT,004000,REGISTRY_INVALID_SEGMENT,Registry record is inactive (status='I')
2,828d5971,GLSegmentType.ACCOUNT,310000,REGISTRY_INVALID_SEGMENT,Registry record is inactive (status='I')
3,828d5971,GLSegmentType.SUB_ACCOUNT,003000,REGISTRY_INVALID_SEGMENT,Registry record is inactive (status='I')


In [8]:
break_case = break_cases[1]

display_break_result(agent.analyze(break_case))

2026-09-19 21:55:40,477 | INFO | break_analysis.agent | Analyzing break case | case_id=9ba9301e | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-19 21:55:40,478 | INFO | break_analysis.agent | Invoking LLM | case_id=9ba9301e | round=1
2026-09-19 21:55:42,943 | INFO | break_analysis.agent | LLM invoked | case_id=9ba9301e | round=1 | tool_calls=1 | input_tokens=1017 | output_tokens=50 | total_tokens=1067 | prompt_eval_count=1017 | eval_count=50 | load_ms=127 | prompt_eval_ms=301 | eval_ms=1775 | total_ms=2464 | duration_ms=2466
2026-09-19 21:55:42,944 | INFO | break_analysis.agent | Tool round | case_id=9ba9301e | round=1 | tool_calls=1
2026-09-19 21:55:43,030 | INFO | break_analysis.agent | Tool invoked | case_id=9ba9301e | tool=validate_segment | args={"business_dt": "2026-03-31", "segment_type": "GL_ACCOUNT", "segment_value": "210000"} | duration_ms=85
2026-09-19 21:55:43,030 | INFO | break_analysis.agent | Invoking LLM | case_id=9ba9301e | round=2
2026-09-19 21:55

## Break Case `9ba9301e` — ✅ **EXPLAINED**

**Records investigated (2):** 8b947b0d, a8887fa1

**Explanation:** The GL_ACCOUNT '210000' in the investigation record is invalid because its registry record exists but is inactive. This explains the $65,000 discrepancy between interface and GL balances.

**Findings (1):**

,recon_result_id,segment_type,segment_value,root_cause,explanation
0,a8887fa1,GLSegmentType.ACCOUNT,210000,REGISTRY_INVALID_SEGMENT,Registry record exists but is marked as inacti...


In [9]:
break_case = break_cases[2]

display_break_result(agent.analyze(break_case))

2026-09-19 21:56:06,464 | INFO | break_analysis.agent | Analyzing break case | case_id=5398a686 | workflow_run_id=d3282a32 | topology=ONE_TO_ONE | records=2
2026-09-19 21:56:06,464 | INFO | break_analysis.agent | Invoking LLM | case_id=5398a686 | round=1
2026-09-19 21:56:18,757 | INFO | break_analysis.agent | LLM invoked | case_id=5398a686 | round=1 | tool_calls=3 | input_tokens=1020 | output_tokens=307 | total_tokens=1327 | prompt_eval_count=1020 | eval_count=307 | load_ms=130 | prompt_eval_ms=310 | eval_ms=11689 | total_ms=12291 | duration_ms=12292
2026-09-19 21:56:18,757 | INFO | break_analysis.agent | Tool round | case_id=5398a686 | round=1 | tool_calls=3
2026-09-19 21:56:18,842 | INFO | break_analysis.services.recon | Break record resolved from recon.result | recon_result_id=4424b323 | workflow_run_id=d3282a32


2026-09-19 21:56:19,946 | INFO | break_analysis.services.atlas | Atlas mapping input values retrieved | recon_result_id=4424b323 | mapping_name=ACCOUNT_TB_MAPPING | input_columns=('SRC_APP_CD', 'SRC_ENTITY_CD', 'DATACLASS', 'COA_RULE_ID', 'SRC_ACCOUNT_ID') | distinct_combinations=1


2026-09-19 21:56:21,806 | INFO | break_analysis.agent | Tool invoked | case_id=5398a686 | tool=investigate_resolution | args={"workflow_run_id": "d3282a32-5edc-4d85-b48c-e0ea38d9a4de", "recon_result_id": "4424b323-4067-433d-a23f-b534b6cf4a2e", "segment_type": "GL_ACCOUNT"} | duration_ms=3048
2026-09-19 21:56:21,881 | INFO | break_analysis.services.recon | Break record resolved from recon.result | recon_result_id=4424b323 | workflow_run_id=d3282a32
2026-09-19 21:56:22,261 | INFO | break_analysis.services.atlas | Atlas mapping input values retrieved | recon_result_id=4424b323 | mapping_name=ACCOUNT_TB_MAPPING | input_columns=('SRC_APP_CD', 'SRC_ENTITY_CD', 'DATACLASS', 'COA_RULE_ID', 'SRC_ACCOUNT_ID') | distinct_combinations=1
2026-09-19 21:56:22,955 | INFO | break_analysis.agent | Tool invoked | case_id=5398a686 | tool=investigate_resolution | args={"workflow_run_id": "d3282a32-5edc-4d85-b48c-e0ea38d9a4de", "recon_result_id": "4424b323-4067-433d-a23f-b534b6cf4a2e", "segment_type": "GL_S

## Break Case `5398a686` — ✅ **EXPLAINED**

**Records investigated (2):** a0498239, 4424b323

**Explanation:** All three segments (GL_ACCOUNT, GL_SUB_ACCOUNT, GL_PRODUCT) in the investigation record were blank and unresolved by Atlas due to no active mapping candidates being found during resolution.

**Findings (3):**

,recon_result_id,segment_type,segment_value,root_cause,explanation
0,4424b323,GLSegmentType.ACCOUNT,(blank),ATLAS_UNRESOLVED_SEGMENT,Segment was blank and no active mapping candid...
1,4424b323,GLSegmentType.SUB_ACCOUNT,(blank),ATLAS_UNRESOLVED_SEGMENT,Segment was blank and no active mapping candid...
2,4424b323,GLSegmentType.PRODUCT,(blank),ATLAS_UNRESOLVED_SEGMENT,Segment was blank and no active mapping candid...


In [10]:
# spark.stop()